In [1]:
import os
import re
import json
import random
import pandas as pd

random.seed(42)

DATA_PATH = "/kaggle/input/datasets/atharvjairath/empathetic-dialogues-facebook-ai/emotion-emotion_69k.csv"
OUTPUT_DIR = "/kaggle/working/empathetic_chatbot_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("_comma_", ",")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_text(x):
    return clean_text(x).lower()

def extract_last_customer_utterance(text):
    text = str(text) if not pd.isna(text) else ""
    text = text.replace("\r", "\n")

    # capture all Customer: ... segments until next speaker tag
    matches = re.findall(
        r"Customer\s*:\s*(.*?)(?=(?:Customer|Agent)\s*:|$)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if matches:
        return clean_text(matches[-1])

    # fallback: remove speaker tags
    text = re.sub(r"(Customer|Agent)\s*:\s*", "", text, flags=re.IGNORECASE)
    return clean_text(text)

def clean_reply(text):
    text = clean_text(text)
    text = re.sub(r"^(Customer|Agent)\s*:\s*", "", text, flags=re.IGNORECASE)
    return clean_text(text)

def looks_bad_reply(text):
    t = normalize_text(text)
    if not t:
        return True
    if len(t.split()) < 3:
        return True
    if len(t.split()) > 30:
        return True
    if any(x in t for x in ["http", "www.", "@", "#"]):
        return True
    return False

def self_focused_reply(text):
    t = normalize_text(text)

    bad_prefixes = (
        "i am ", "i'm ", "i was ", "i have ", "i had ",
        "my ", "me and ", "when i ", "i remember ",
        "i went ", "i feel ", "my grandparents", "my parents"
    )

    return t.startswith(bad_prefixes)

def empathetic_reply(text):
    t = normalize_text(text)

    good_patterns = [
        "that sounds",
        "i'm sorry",
        "i am sorry",
        "are you okay",
        "what happened",
        "how are you",
        "do you want",
        "that must be",
        "i understand",
        "that seems",
        "glad to hear",
        "i'm glad",
        "i am glad",
        "you must",
        "that is hard",
        "that sounds hard",
        "that sounds difficult"
    ]
    return any(p in t for p in good_patterns) or "?" in t

work = df.copy()
work["emotion"] = work["emotion"].apply(normalize_text)
work["user_message"] = work["empathetic_dialogues"].apply(extract_last_customer_utterance)
work["assistant_reply"] = work["labels"].apply(clean_reply)

pairs_df = work[["emotion", "user_message", "assistant_reply"]].drop_duplicates().copy()
pairs_df = pairs_df[
    (pairs_df["emotion"] != "") &
    (pairs_df["user_message"] != "") &
    (pairs_df["assistant_reply"] != "")
].copy()

pairs_df["bad_reply"] = pairs_df["assistant_reply"].apply(looks_bad_reply)
pairs_df["self_focused"] = pairs_df["assistant_reply"].apply(self_focused_reply)
pairs_df["empathetic"] = pairs_df["assistant_reply"].apply(empathetic_reply)

filtered_df = pairs_df[
    (~pairs_df["bad_reply"]) &
    (~pairs_df["self_focused"]) &
    (pairs_df["empathetic"])
].copy()

print("Raw pairs:", len(pairs_df))
print("Filtered pairs:", len(filtered_df))
print(filtered_df.head(10))

response_bank = {}
for emotion, g in filtered_df.groupby("emotion"):
    replies = g["assistant_reply"].drop_duplicates().tolist()
    random.shuffle(replies)
    response_bank[emotion] = replies[:150]

pairs_path = os.path.join(OUTPUT_DIR, "training_pairs_clean.csv")
bank_path = os.path.join(OUTPUT_DIR, "response_bank_clean.json")

filtered_df.to_csv(pairs_path, index=False)

with open(bank_path, "w", encoding="utf-8") as f:
    json.dump(response_bank, f, ensure_ascii=False, indent=2)

print("Saved:", pairs_path)
print("Saved:", bank_path)

for mood in ["sad", "anxious", "joyful", "angry", "sentimental"]:
    sample = response_bank.get(mood, [])[:3]
    print(f"\n{mood}:")
    for s in sample:
        print("-", s)


Raw pairs: 64620
Filtered pairs: 16661
        emotion                                       user_message  \
0   sentimental  I remember going to see the fireworks with my ...   
1   sentimental                This was a best friend. I miss her.   
2   sentimental                                 We no longer talk.   
5        afraid  it feels like hitting to blank wall when i see...   
15     faithful   Yea it hasn't been easy but I am proud I haven't   
20    terrified            Yes but if you stay calm it will be ok.   
21       joyful  Hi, this year, I was the first over 300 studen...   
27          sad  During christmas a few years ago, I did not ge...   
28          sad  Since that day christmas has not been a good t...   
41     prepared  I had to board my entire house when the hurric...   

                                      assistant_reply  bad_reply  \
0   Was this a friend you were in love with, or ju...      False   
1                                 Where has she gone? 